In [0]:
%pip install great-expectations==0.18.21

  Using cached great_expectations-0.18.21-py3-none-any.whl.metadata (8.5 kB)
  Using cached altair-4.2.2-py3-none-any.whl.metadata (13 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached makefun-1.16.0-py2.py3-none-any.whl.metadata (2.9 kB)
  Using cached ruamel.yaml-0.17.40-py3-none-any.whl.metadata (19 kB)
  Using cached tzlocal-5.3.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached entrypoints-0.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached toolz-1.1.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached ruamel_yaml_clib-0.2.15-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (3.5 kB)
Using cached great_expectations-0.18.21-py3-none-any.whl (5.4 MB)
Using cached altair-4.2.2-py3-none-any.whl (813 kB)
Using cached colorama-0.4.6-py2.py3-none-any.whl (25 kB)
Using cached makefun-1.16.0-py2.py3-none-any.w

## Silver Great Expectations
Validates the canonical Silver `header` and `line_items` model after Bronze ingestion and Silver transformation.

In [0]:
import json
import great_expectations as gx
from great_expectations.checkpoint import Checkpoint

from datetime import date, datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.functions import col, lit, coalesce, when

In [0]:
# VALID_SILVER_HEADER_PATH = dbutils.jobs.taskValues.get(
#     taskKey="silver_transform",
#     key="valid_silver_header_path",
#     debugValue=""
# )
# VALID_SILVER_LINES_PATH = dbutils.jobs.taskValues.get(
#     taskKey="silver_transform",
#     key="valid_silver_lines_path",
#     debugValue=""
# )

STORAGE_ACCOUNT = "hantstorageaccount"
LAKEHOUSE_CONTAINER = "lakehouse"

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "invoice"

VALID_SILVER_HEADER_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/transform/standardized/header/"
VALID_SILVER_LINES_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/transform/standardized/lines/"

GE_VALID_SILVER_HEADER_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/validated/header/"
GE_VALID_SILVER_LINES_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/validated/lines/"
GE_QUARANTINE_SILVER_HEADER_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/ge/quarantine/header/"
GE_QUARANTINE_SILVER_LINES_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/ge/quarantine/lines/"
GE_WARNING_SILVER_HEADER_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/ge/warning/header/"
GE_WARNING_SILVER_LINES_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/ge/warning/lines/"
GE_ISSUE_LOG_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/ge/issue_log/"

# GE silver_ge_context + Data Docs config
CONTEXT_ROOT_DIR = "/dbfs/tmp/ge/silver"
DATA_DOCS_SITE_NAME = "silver_local_site"
PIPELINE_LAYER = "silver"

# Build full Data Docs site directly here so /files/... works
FILESTORE_DATA_DOCS_DBFS_PATH = "dbfs:/FileStore/great_expectations/silver/"
FILESTORE_DATA_DOCS_DBFS_FUSE_PATH = "/dbfs/FileStore/great_expectations/silver"
FILESTORE_DATA_DOCS_BROWSER_PATH = "/files/great_expectations/silver/index.html"

# Persist full site to ADLS too
STORAGE_DATA_DOCS_LATEST_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/monitoring/data_docs/silver/latest/"
STORAGE_DATA_DOCS_RUNS_ROOT = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/monitoring/data_docs/silver/runs/"

GE_RUNTIME_METRICS_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/monitoring/ge_runtime_metrics/"
GE_RUNTIME_METRICS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_ge_runtime_metrics"

SILVER_HEADER_COLUMNS = [
    "_source_file",
    "source_type",
    "InvoiceId",
    "OrderDate",
    "CustomerName",
    "ShipPostalCode",
    "ShipCity",
    "ShipState",
    "ShipCountry",
    "ShipMode",
    "BalanceDue",
    "SubTotal",
    "DiscountPercent",
    "DiscountAmount",
    "ShippingAmount",
    "InvoiceTotal",
    "OrderId",
]

SILVER_LINES_COLUMNS = [
    "_source_file",
    "source_type",
    "InvoiceId",
    "LineNumber",
    "ProductName",
    "SubCategory",
    "Category",
    "ProductId",
    "Quantity",
    "UnitPrice",
    "ItemSubTotal",
]

SILVER_HEADER_ARITH_COLUMNS = [
    "_source_file",
    "source_type",
    "InvoiceId",
    "SubTotal",
    "DiscountAmount",
    "ShippingAmount",
    "InvoiceTotal",
    "_line_count",
    "_sum_item_subtotal",
    "_sum_calc_item_subtotal",
    "_sum_line_diffs",
    "_calc_invoice_total",
    "_header_total_diff",
    "_line_to_subtotal_diff",
]

SILVER_LINES_ARITH_COLUMNS = [
    "_source_file",
    "source_type",
    "InvoiceId",
    "LineNumber",
    "Quantity",
    "UnitPrice",
    "ItemSubTotal",
    "_calc_item_subtotal",
    "_line_subtotal_diff",
]

SILVER_SOURCE_TYPES = ["csv", "json"]
SILVER_SHIP_MODES = ["FIRST CLASS", "SECOND CLASS", "STANDARD CLASS", "SAME DAY"]
SILVER_ORDER_DATE_MIN = date(2010, 1, 1)
SILVER_ARITH_TOLERANCE_LINE = 0.01
SILVER_ARITH_TOLERANCE_HEADER = 0.05
SILVER_WARNING_UPPER_BOUND_LINE = 0.05
SILVER_WARNING_UPPER_BOUND_HEADER = 1.00
SILVER_ENABLE_ARITH_PIPELINE_GATE = False
SILVER_GATE_MAX_ERROR_INVOICES = 0
SILVER_GATE_MAX_ERROR_RATE = 0.01

ISSUE_SCHEMA = T.StructType(
    [
        T.StructField("layer", T.StringType(), False),
        T.StructField("dataset", T.StringType(), False),
        T.StructField("rule_id", T.StringType(), False),
        T.StructField("severity", T.StringType(), False),
        T.StructField("_source_file", T.StringType(), True),
        T.StructField("source_type", T.StringType(), True),
        T.StructField("InvoiceId", T.StringType(), True),
        T.StructField("LineNumber", T.StringType(), True),
        T.StructField("issue_type", T.StringType(), False),
        T.StructField("dq_reason", T.StringType(), False),
        T.StructField("issue_ts", T.TimestampType(), False),
    ]
)

GE_RUNTIME_SCHEMA = T.StructType([
    T.StructField("layer", T.StringType(), False),
    T.StructField("suite_name", T.StringType(), False),
    T.StructField("run_id", T.StringType(), True),
    T.StructField("started_at", T.TimestampType(), False),
    T.StructField("ended_at", T.TimestampType(), False),
    T.StructField("runtime_seconds", T.DoubleType(), False),
    T.StructField("rows_evaluated", T.LongType(), True),
    T.StructField("rows_per_second", T.DoubleType(), True),
    T.StructField("evaluated_expectations", T.LongType(), True),
    T.StructField("successful_expectations", T.LongType(), True),
    T.StructField("failed_expectations", T.LongType(), True),
    T.StructField("expectation_success_percent", T.DoubleType(), True),
    T.StructField("validation_success", T.BooleanType(), True),
    T.StructField("recorded_at", T.TimestampType(), False),
])

In [0]:
def get_gx_context():
    return gx.get_context(context_root_dir=CONTEXT_ROOT_DIR)


def get_or_create_spark_datasource(silver_ge_context, datasource_name="spark_datasource"):
    try:
        return silver_ge_context.sources.add_or_update_spark(name=datasource_name)
    except AttributeError:
        context.add_datasource(
            datasource_name,
            class_name="Datasource",
            execution_engine={"class_name": "SparkDFExecutionEngine"},
            data_connectors={
                "runtime_data_connector": {
                    "class_name": "RuntimeDataConnector",
                    "batch_identifiers": ["default_identifier_name"],
                }
            },
        )
        return context.get_datasource(datasource_name)


def build_batch_request(datasource, asset_name, dataframe):
    asset = datasource.add_dataframe_asset(name=asset_name)
    return asset.build_batch_request(dataframe=dataframe)

def ensure_data_docs_site(context):
    try:
        context.delete_data_docs_site(DATA_DOCS_SITE_NAME)
    except Exception:
        pass

    context.add_data_docs_site(
        site_name=DATA_DOCS_SITE_NAME,
        site_config={
            "class_name": "SiteBuilder",
            "store_backend": {
                "class_name": "TupleFilesystemStoreBackend",
                "base_directory": FILESTORE_DATA_DOCS_DBFS_FUSE_PATH,
            },
            "site_index_builder": {"class_name": "DefaultSiteIndexBuilder"},
        },
    )

def add_or_update_checkpoint(context, checkpoint_name, batch_request, suite_name):
    checkpoint = Checkpoint(
        name=checkpoint_name,
        data_context=context,
        validations=[
            {
                "batch_request": batch_request,
                "expectation_suite_name": suite_name,
            }
        ],
        action_list=[
            {
                "name": "store_validation_result",
                "action": {"class_name": "StoreValidationResultAction"},
            },
            {
                "name": "update_data_docs",
                "action": {"class_name": "UpdateDataDocsAction"},
            },
        ],
        run_name_template="%Y%m%dT%H%M%S_silver_ge",
    )
    context.add_or_update_checkpoint(checkpoint=checkpoint)
    return checkpoint

def _safe_rm(path: str):
    try:
        dbutils.fs.rm(path, True)
    except Exception:
        pass


def _safe_ls(path: str, label: str):
    print(f"\n{label}: {path}")
    try:
        for x in dbutils.fs.ls(path):
            print(f" - {x.path}")
    except Exception as e:
        print(f"   [unable to list] {e}")


def publish_data_docs(context, layer_name=PIPELINE_LAYER):
    _safe_rm(FILESTORE_DATA_DOCS_DBFS_PATH)

    context.build_data_docs(site_names=[DATA_DOCS_SITE_NAME])

    _safe_ls(FILESTORE_DATA_DOCS_DBFS_PATH, "Data Docs root")
    _safe_ls(FILESTORE_DATA_DOCS_DBFS_PATH + "expectations/", "Data Docs expectations")
    _safe_ls(FILESTORE_DATA_DOCS_DBFS_PATH + "validations/", "Data Docs validations")

    run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    storage_run_path = f"{STORAGE_DATA_DOCS_RUNS_ROOT}{run_stamp}/"

    _safe_rm(STORAGE_DATA_DOCS_LATEST_PATH)
    dbutils.fs.mkdirs(STORAGE_DATA_DOCS_LATEST_PATH)
    dbutils.fs.mkdirs(storage_run_path)

    for item in dbutils.fs.ls(FILESTORE_DATA_DOCS_DBFS_PATH):
        dbutils.fs.cp(item.path, STORAGE_DATA_DOCS_LATEST_PATH + item.name, True)
        dbutils.fs.cp(item.path, storage_run_path + item.name, True)

    _safe_ls(STORAGE_DATA_DOCS_LATEST_PATH, "ADLS Data Docs latest")
    _safe_ls(storage_run_path, "ADLS Data Docs run")

    return {
        "preview_url": FILESTORE_DATA_DOCS_BROWSER_PATH,
        "storage_latest_path": STORAGE_DATA_DOCS_LATEST_PATH,
        "storage_run_path": storage_run_path,
    }


def render_data_docs_preview(data_docs_info, title):
    displayHTML(
        f'''
        <div style="margin:16px 0;">
          <p><strong>{title}</strong></p>
          <p><a href="{data_docs_info["preview_url"]}" target="_blank">Open Data Docs index</a></p>
          <iframe
            src="{data_docs_info["preview_url"]}"
            width="100%"
            height="900"
            style="border:1px solid #d0d7de;border-radius:8px;background:#fff;">
          </iframe>
        </div>
        '''
    )
    
def empty_issue_df():
    return spark.createDataFrame([], ISSUE_SCHEMA)


def issue_events_from_condition(df, condition, dataset, rule_id, severity, issue_type, dq_reason, line_col=None):
    out_df = df.filter(condition)
    return (
        out_df.select(
            F.lit("silver").alias("layer"),
            F.lit(dataset).alias("dataset"),
            F.lit(rule_id).alias("rule_id"),
            F.lit(severity).alias("severity"),
            F.col("_source_file").cast("string").alias("_source_file"),
            F.col("source_type").cast("string").alias("source_type"),
            F.col("InvoiceId").cast("string").alias("InvoiceId"),
            (F.col(line_col).cast("string") if line_col else F.lit(None).cast("string")).alias("LineNumber"),
            F.lit(issue_type).alias("issue_type"),
            F.lit(dq_reason).alias("dq_reason"),
            F.current_timestamp().alias("issue_ts"),
        ).dropDuplicates()
    )


def extract_first_validation(checkpoint_result):
    run_results = checkpoint_result.run_results
    first_key = list(run_results.keys())[0]
    return run_results[first_key]["validation_result"]


def summarize_result(result, suite_name: str) -> dict:
    stats = result.get("statistics", {}) or {}

    failures = [
        {
            "expectation": (r.get("expectation_config", {}) or {}).get("expectation_type"),
            "column": ((r.get("expectation_config", {}) or {}).get("kwargs", {}) or {}).get("column", "-"),
            "kwargs": (r.get("expectation_config", {}) or {}).get("kwargs", {}) or {},
            "result": r.get("result", {}) or {},
            "exception": ((r.get("exception_info", {}) or {}).get("exception_message")),
        }
        for r in result.get("results", [])
        if not r.get("success", False)
    ]

    return {
        "suite": suite_name,
        "success": result.get("success"),
        "evaluated": stats.get("evaluated_expectations", 0),
        "successful": stats.get("successful_expectations", 0),
        "failed": stats.get("unsuccessful_expectations", 0),
        "success_pct": stats.get("success_percent"),
        "failures": failures,
    }


def display_summary(summary: dict):
    status = "PASSED" if summary["success"] else "FAILED"
    success_pct = summary.get("success_pct")
    score_text = f"{success_pct:.1f}%" if success_pct is not None else "N/A"

    print(f"\n{'=' * 65}")
    print(f"  Suite : {summary['suite']}")
    print(f"  Status: {status}")
    print(
        f"  Evaluated : {summary['evaluated']}  |  "
        f"Passed: {summary['successful']}  |  "
        f"Failed: {summary['failed']}  |  "
        f"Score: {score_text}"
    )
    print(f"{'=' * 65}")

    if summary["failures"]:
        print("  -- Failed Expectations --")
        for failure in summary["failures"]:
            print(f"   [{failure['column']}] {failure['expectation']}")
            if failure["exception"]:
                print(f"      Exception: {failure['exception']}")
            elif failure["result"]:
                print(f"      Result: {json.dumps(failure['result'], default=str)[:200]}")
    print()


def build_result_format(key_columns):
    return {
        "result_format": "COMPLETE",
        "unexpected_index_column_names": key_columns,
        "return_unexpected_index_query": True,
    }


def build_issue_meta(rule_id, severity, dataset, issue_type, dq_reason, key_columns):
    return {
        "rule_id": rule_id,
        "severity": severity,
        "dataset": dataset,
        "issue_type": issue_type,
        "dq_reason": dq_reason,
        "key_columns": key_columns,
    }


def extract_issue_rows_from_validation_result(validation_result):
    issue_rows = []
    for result in validation_result.get("results", []):
        if result.get("success", True):
            continue
        expectation_config = result.get("expectation_config", {}) or {}
        meta = expectation_config.get("meta", {}) or {}
        rule_id = meta.get("rule_id")
        severity = meta.get("severity")
        if not rule_id or not severity:
            continue

        payload = result.get("result", {}) or {}
        unexpected_index_list = payload.get("unexpected_index_list") or []
        if not unexpected_index_list:
            issue_rows.append(
                {
                    "layer": "silver",
                    "dataset": meta.get("dataset"),
                    "rule_id": rule_id,
                    "severity": severity,
                    "_source_file": None,
                    "source_type": None,
                    "InvoiceId": None,
                    "LineNumber": None,
                    "issue_type": meta.get("issue_type"),
                    "dq_reason": meta.get("dq_reason"),
                    "issue_ts": datetime.now(timezone.utc),
                }
            )
            continue

        for item in unexpected_index_list:
            row = {
                "layer": "silver",
                "dataset": meta.get("dataset"),
                "rule_id": rule_id,
                "severity": severity,
                "_source_file": None,
                "source_type": None,
                "InvoiceId": None,
                "LineNumber": None,
                "issue_type": meta.get("issue_type"),
                "dq_reason": meta.get("dq_reason"),
                "issue_ts": datetime.now(timezone.utc),
            }
            if isinstance(item, dict):
                row["_source_file"] = None if item.get("_source_file") is None else str(item.get("_source_file"))
                row["source_type"] = None if item.get("source_type") is None else str(item.get("source_type"))
                row["InvoiceId"] = None if item.get("InvoiceId") is None else str(item.get("InvoiceId"))
                row["LineNumber"] = None if item.get("LineNumber") is None else str(item.get("LineNumber"))
            else:
                key_columns = meta.get("key_columns") or []
                if len(key_columns) == 1:
                    row[key_columns[0]] = None if item is None else str(item)
            issue_rows.append(row)
    return issue_rows


def issue_df_from_validation_results(validation_results):
    rows = []
    for validation_result in validation_results:
        rows.extend(extract_issue_rows_from_validation_result(validation_result))
    if not rows:
        return empty_issue_df()
    return spark.createDataFrame(rows, ISSUE_SCHEMA).dropDuplicates()

In [0]:
# runtime metrics helper functions
def first_validation_result(checkpoint_result):
    run_results = checkpoint_result.run_results
    return run_results[list(run_results.keys())[0]]["validation_result"]


def validation_statistics(validation_result):
    stats = validation_result.get("statistics", {}) or {}
    evaluated = int(stats.get("evaluated_expectations", 0) or 0)
    successful = int(stats.get("successful_expectations", 0) or 0)
    failed = int(stats.get("unsuccessful_expectations", 0) or 0)
    success_percent = float(successful / evaluated) if evaluated else None
    return evaluated, successful, failed, success_percent


def write_ge_runtime_metric(layer, suite_name, run_id, started_at, ended_at, rows_evaluated, validation_result):
    runtime_seconds = (ended_at - started_at).total_seconds()
    evaluated, successful, failed, success_percent = validation_statistics(validation_result)
    rows_per_second = float(rows_evaluated / runtime_seconds) if rows_evaluated and runtime_seconds > 0 else None

    row = [(
        layer,
        suite_name,
        run_id,
        started_at,
        ended_at,
        float(runtime_seconds),
        int(rows_evaluated) if rows_evaluated is not None else None,
        rows_per_second,
        evaluated,
        successful,
        failed,
        success_percent,
        bool(validation_result.get("success")),
        datetime.now(timezone.utc),
    )]

    runtime_df = spark.createDataFrame(row, GE_RUNTIME_SCHEMA)
    runtime_df.write.format("delta").mode("append").save(GE_RUNTIME_METRICS_PATH)

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")
    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {GE_RUNTIME_METRICS_TABLE}
        USING DELTA
        LOCATION "{GE_RUNTIME_METRICS_PATH}"
    ''')

In [0]:
def build_silver_lines_arithmetic_df(silver_lines_df):
    return (
        silver_lines_df
        .select(*SILVER_LINES_COLUMNS)
        .withColumn(
            "_calc_item_subtotal",
            when(
                col("Quantity").isNotNull() & col("UnitPrice").isNotNull(),
                (col("Quantity") * col("UnitPrice")).cast("decimal(18,2)")
            ).otherwise(None)
        )
        .withColumn(
            "_line_subtotal_diff",
            when(
                col("ItemSubTotal").isNotNull() & col("_calc_item_subtotal").isNotNull(),
                (col("ItemSubTotal") - col("_calc_item_subtotal")).cast("decimal(18,2)")
            ).otherwise(None)
        )
    )


def build_silver_header_arithmetic_df(silver_header_df, silver_lines_df):
    silver_lines_arith_df = build_silver_lines_arithmetic_df(silver_lines_df)

    lines_aggregates = (
        silver_lines_arith_df
        .groupBy("InvoiceId")
        .agg(
            F.count("LineNumber").cast("int").alias("_line_count"),
            F.sum("ItemSubTotal").cast("decimal(18,2)").alias("_sum_item_subtotal"),
            F.sum("_calc_item_subtotal").cast("decimal(18,2)").alias("_sum_calc_item_subtotal"),
            F.sum("_line_subtotal_diff").cast("decimal(18,2)").alias("_sum_line_diffs"),
        )
    )

    return (
        silver_header_df
        .select(*SILVER_HEADER_COLUMNS)
        .join(lines_aggregates, on="InvoiceId", how="left")
        .withColumn(
            "_calc_invoice_total",
            when(
                col("SubTotal").isNotNull(),
                (
                    col("SubTotal")
                    - coalesce(col("DiscountAmount"), F.lit(0).cast("decimal(18,2)"))
                    + coalesce(col("ShippingAmount"), F.lit(0).cast("decimal(18,2)"))
                ).cast("decimal(18,2)")
            ).otherwise(None)
        )
        .withColumn(
            "_header_total_diff",
            when(
                col("InvoiceTotal").isNotNull() & col("_calc_invoice_total").isNotNull(),
                (col("InvoiceTotal") - col("_calc_invoice_total")).cast("decimal(18,2)")
            ).otherwise(None)
        )
        .withColumn(
            "_line_to_subtotal_diff",
            when(
                col("SubTotal").isNotNull() & col("_sum_item_subtotal").isNotNull(),
                (col("SubTotal") - col("_sum_item_subtotal")).cast("decimal(18,2)")
            ).otherwise(None)
        )
    )

In [0]:
silver_ge_context = get_gx_context()

def run_silver_header_validation(silver_header_df, include_warning_rules=True, run_label="pre_route"):
    datasource = get_or_create_spark_datasource(silver_ge_context, f"silver_header_datasource_{run_label}")
    batch_request = build_batch_request(
        datasource=datasource,
        asset_name=f"silver_header_asset_{run_label}",
        dataframe=silver_header_df.select(*SILVER_HEADER_COLUMNS),
    )
    suite_name = f"silver_header_suite_{run_label}"
    checkpoint_name = f"silver_header_checkpoint_{run_label}"
    silver_ge_context.add_or_update_expectation_suite(expectation_suite_name=suite_name)
    validator = silver_ge_context.get_validator(batch_request=batch_request, expectation_suite_name=suite_name)

    header_keys = ["InvoiceId", "_source_file", "source_type"]

    for col_name in SILVER_HEADER_COLUMNS:
        validator.expect_column_to_exist(col_name)

    validator.expect_column_values_to_be_in_set(
        "source_type",
        SILVER_SOURCE_TYPES,
        result_format=build_result_format(header_keys),
        meta=build_issue_meta("silver_header_source_type_domain", "ERROR", "header", "contract", "source_type is outside the allowed domain", header_keys),
    )
    validator.expect_column_values_to_be_in_set(
        "ShipMode",
        SILVER_SHIP_MODES,
        mostly=0.99,
        result_format=build_result_format(header_keys),
        meta=build_issue_meta("silver_header_ship_mode_domain", "ERROR", "header", "contract", "ShipMode is missing or outside the allowed domain", header_keys),
    )

    for col_name, type_name, rule_id in [
        ("_source_file", "StringType", "silver_header_source_file_type"),
        ("source_type", "StringType", "silver_header_source_type_type"),
        ("InvoiceId", "StringType", "silver_header_invoice_id_type"),
        ("OrderDate", "DateType", "silver_header_order_date_type"),
        ("CustomerName", "StringType", "silver_header_customer_name_type"),
        ("ShipPostalCode", "StringType", "silver_header_ship_postal_code_type"),
        ("ShipCity", "StringType", "silver_header_ship_city_type"),
        ("ShipState", "StringType", "silver_header_ship_state_type"),
        ("ShipCountry", "StringType", "silver_header_ship_country_type"),
        ("ShipMode", "StringType", "silver_header_ship_mode_type"),
        ("OrderId", "StringType", "silver_header_order_id_type"),
        ("BalanceDue", "DecimalType", "silver_header_balance_due_type"),
        ("SubTotal", "DecimalType", "silver_header_subtotal_type"),
        ("DiscountPercent", "DecimalType", "silver_header_discount_percent_type"),
        ("DiscountAmount", "DecimalType", "silver_header_discount_amount_type"),
        ("ShippingAmount", "DecimalType", "silver_header_shipping_amount_type"),
        ("InvoiceTotal", "DecimalType", "silver_header_invoice_total_type"),
    ]:
        validator.expect_column_values_to_be_of_type(
            col_name,
            type_name,
            result_format=build_result_format(header_keys),
            meta=build_issue_meta(rule_id, "ERROR", "header", "contract", f"{col_name} has an unexpected type", header_keys),
        )

    for col_name, rule_id, reason in [
        ("_source_file", "silver_header_source_file_present", "_source_file is missing in Silver header"),
        ("InvoiceId", "silver_header_invoice_id_present", "InvoiceId is missing in Silver header"),
        ("OrderDate", "silver_header_order_date_present", "OrderDate is missing in Silver header"),
        ("CustomerName", "silver_header_customer_name_present", "CustomerName is missing in Silver header"),
        ("ShipMode", "silver_header_ship_mode_present", "ShipMode is missing in Silver header"),
        ("SubTotal", "silver_header_subtotal_present", "SubTotal is missing in Silver header"),
        ("InvoiceTotal", "silver_header_invoice_total_present", "InvoiceTotal is missing in Silver header"),
        ("source_type", "silver_header_source_type_present", "source_type is missing in Silver header"),
    ]:
        validator.expect_column_values_to_not_be_null(
            col_name,
            result_format=build_result_format(header_keys),
            meta=build_issue_meta(rule_id, "ERROR", "header", "contract", reason, header_keys),
        )

    validator.expect_column_values_to_be_unique(
        "InvoiceId",
        result_format=build_result_format(header_keys),
        meta=build_issue_meta("silver_header_invoice_id_unique", "ERROR", "header", "contract", "InvoiceId is duplicated in Silver header", header_keys),
    )

    validator.expect_column_values_to_be_between(
        "OrderDate",
        min_value=SILVER_ORDER_DATE_MIN,
        max_value=date.today(),
        parse_strings_as_datetimes=True,
        result_format=build_result_format(header_keys),
        meta=build_issue_meta("silver_header_order_date_valid", "ERROR", "header", "contract", "OrderDate is outside the allowed range", header_keys),
    )

    for col_name, rule_id, reason in [
        ("SubTotal", "silver_header_subtotal_non_negative", "SubTotal is negative in Silver header"),
        ("InvoiceTotal", "silver_header_invoice_total_non_negative", "InvoiceTotal is negative in Silver header"),
        ("BalanceDue", "silver_header_balance_due_non_negative", "BalanceDue is negative in Silver header"),
        ("DiscountAmount", "silver_header_discount_amount_non_negative", "DiscountAmount is negative in Silver header"),
        ("ShippingAmount", "silver_header_shipping_amount_non_negative", "ShippingAmount is negative in Silver header"),
    ]:
        validator.expect_column_values_to_be_between(
            col_name,
            min_value=0,
            max_value=None,
            mostly=0.99,
            result_format=build_result_format(header_keys),
            meta=build_issue_meta(rule_id, "ERROR", "header", "contract", reason, header_keys),
        )

    validator.expect_column_values_to_be_between(
        "DiscountPercent",
        min_value=0,
        max_value=1,
        mostly=0.99,
        result_format=build_result_format(header_keys),
        meta=build_issue_meta("silver_header_discount_percent_range", "ERROR", "header", "contract", "DiscountPercent is outside [0, 1]", header_keys),
    )

    if include_warning_rules:
        for col_name, rule_id, reason in [
            ("ShipPostalCode", "silver_header_ship_postal_code_present_warning", "ShipPostalCode is missing in Silver header"),
            ("ShipState", "silver_header_ship_state_present_warning", "ShipState is missing in Silver header"),
            ("ShipCity", "silver_header_ship_city_present_warning", "ShipCity is missing in Silver header"),
            ("ShipCountry", "silver_header_ship_country_present_warning", "ShipCountry is missing in Silver header"),
        ]:
            validator.expect_column_values_to_not_be_null(
                col_name,
                result_format=build_result_format(header_keys),
                meta=build_issue_meta(rule_id, "WARNING", "header", "contract", reason, header_keys),
            )

    validator.save_expectation_suite(discard_failed_expectations=False)
    add_or_update_checkpoint(silver_ge_context, checkpoint_name, batch_request, suite_name)
    start_at = datetime.now(timezone.utc)
    checkpoint_result = silver_ge_context.run_checkpoint(checkpoint_name=checkpoint_name)
    ended_at = datetime.now(timezone.utc)
    validation_result = extract_first_validation(checkpoint_result)
    write_ge_runtime_metric(
        layer="silver",
        suite_name=suite_name,
        run_id=checkpoint_result.run_id,
        started_at=start_at,
        ended_at=ended_at,
        rows_evaluated=silver_header_df.count(),
        validation_result=validation_result
    )
    return checkpoint_result, suite_name


def run_silver_lines_validation(silver_lines_df, include_warning_rules=True, run_label="pre_route"):
    datasource = get_or_create_spark_datasource(silver_ge_context, f"silver_lines_datasource_{run_label}")
    batch_request = build_batch_request(
        datasource=datasource,
        asset_name=f"silver_lines_asset_{run_label}",
        dataframe=silver_lines_df.select(*SILVER_LINES_COLUMNS),
    )
    suite_name = f"silver_lines_suite_{run_label}"
    checkpoint_name = f"silver_lines_checkpoint_{run_label}"
    silver_ge_context.add_or_update_expectation_suite(expectation_suite_name=suite_name)
    validator = silver_ge_context.get_validator(batch_request=batch_request, expectation_suite_name=suite_name)

    line_keys = ["InvoiceId", "LineNumber", "_source_file", "source_type"]

    for col_name in SILVER_LINES_COLUMNS:
        validator.expect_column_to_exist(col_name)

    validator.expect_column_values_to_be_in_set(
        "source_type",
        SILVER_SOURCE_TYPES,
        result_format=build_result_format(line_keys),
        meta=build_issue_meta("silver_lines_source_type_domain", "ERROR", "lines", "contract", "source_type is outside the allowed domain", line_keys),
    )

    for col_name, type_name, rule_id in [
        ("_source_file", "StringType", "silver_lines_source_file_type"),
        ("source_type", "StringType", "silver_lines_source_type_type"),
        ("InvoiceId", "StringType", "silver_lines_invoice_id_type"),
        ("LineNumber", "IntegerType", "silver_lines_line_number_type"),
        ("ProductName", "StringType", "silver_lines_product_name_type"),
        ("SubCategory", "StringType", "silver_lines_subcategory_type"),
        ("Category", "StringType", "silver_lines_category_type"),
        ("ProductId", "StringType", "silver_lines_product_id_type"),
        ("Quantity", "IntegerType", "silver_lines_quantity_type"),
        ("UnitPrice", "DecimalType", "silver_lines_unit_price_type"),
        ("ItemSubTotal", "DecimalType", "silver_lines_item_subtotal_type"),
    ]:
        validator.expect_column_values_to_be_of_type(
            col_name,
            type_name,
            result_format=build_result_format(line_keys),
            meta=build_issue_meta(rule_id, "ERROR", "lines", "contract", f"{col_name} has an unexpected type", line_keys),
        )

    for col_name, rule_id, reason in [
        ("_source_file", "silver_lines_source_file_present", "_source_file is missing in Silver lines"),
        ("InvoiceId", "silver_lines_invoice_id_present", "InvoiceId is missing in Silver lines"),
        ("LineNumber", "silver_lines_line_number_present", "LineNumber is missing in Silver lines"),
        ("ProductName", "silver_lines_product_name_present", "ProductName is missing in Silver lines"),
        ("ProductId", "silver_lines_product_id_present", "ProductId is missing in Silver lines"),
        ("Quantity", "silver_lines_quantity_present", "Quantity is missing in Silver lines"),
        ("UnitPrice", "silver_lines_unit_price_present", "UnitPrice is missing in Silver lines"),
        ("ItemSubTotal", "silver_lines_item_subtotal_present", "ItemSubTotal is missing in Silver lines"),
        ("source_type", "silver_lines_source_type_present", "source_type is missing in Silver lines"),
    ]:
        validator.expect_column_values_to_not_be_null(
            col_name,
            result_format=build_result_format(line_keys),
            meta=build_issue_meta(rule_id, "ERROR", "lines", "contract", reason, line_keys),
        )

    validator.expect_compound_columns_to_be_unique(
        ["InvoiceId", "LineNumber"],
        result_format=build_result_format(line_keys),
        meta=build_issue_meta("silver_lines_compound_key_unique", "ERROR", "lines", "contract", "(InvoiceId, LineNumber) is duplicated in Silver lines", line_keys),
    )
    validator.expect_column_values_to_be_between(
        "LineNumber",
        min_value=1,
        max_value=None,
        result_format=build_result_format(line_keys),
        meta=build_issue_meta("silver_lines_line_number_positive", "ERROR", "lines", "contract", "LineNumber is not positive", line_keys),
    )
    validator.expect_column_values_to_be_between(
        "Quantity",
        min_value=1,
        max_value=None,
        result_format=build_result_format(line_keys),
        meta=build_issue_meta("silver_lines_quantity_positive", "ERROR", "lines", "contract", "Quantity is not positive", line_keys),
    )
    validator.expect_column_values_to_be_between(
        "UnitPrice",
        min_value=0,
        max_value=None,
        strict_min=True,
        result_format=build_result_format(line_keys),
        meta=build_issue_meta("silver_lines_unit_price_positive", "ERROR", "lines", "contract", "UnitPrice is not positive", line_keys),
    )
    validator.expect_column_values_to_be_between(
        "ItemSubTotal",
        min_value=0,
        max_value=None,
        result_format=build_result_format(line_keys),
        meta=build_issue_meta("silver_lines_item_subtotal_non_negative", "ERROR", "lines", "contract", "ItemSubTotal is negative", line_keys),
    )

    if include_warning_rules:
        for col_name, rule_id, reason in [
            ("Category", "silver_lines_category_present_warning", "Category is missing in Silver lines"),
            ("SubCategory", "silver_lines_subcategory_present_warning", "SubCategory is missing in Silver lines"),
        ]:
            validator.expect_column_values_to_not_be_null(
                col_name,
                result_format=build_result_format(line_keys),
                meta=build_issue_meta(rule_id, "WARNING", "lines", "contract", reason, line_keys),
            )

    validator.save_expectation_suite(discard_failed_expectations=False)
    add_or_update_checkpoint(silver_ge_context, checkpoint_name, batch_request, suite_name)
    started_at = datetime.now(timezone.utc)
    checkpoint_result = silver_ge_context.run_checkpoint(checkpoint_name=checkpoint_name)
    ended_at = datetime.now(timezone.utc)
    validation_result = extract_first_validation(checkpoint_result)
    write_ge_runtime_metric(
        layer="silver",
        suite_name=suite_name,
        run_id=checkpoint_result.run_id,
        started_at=started_at,
        ended_at=ended_at,
        rows_evaluated=silver_lines_df.count(),
        validation_result=validation_result
    )
    return checkpoint_result, suite_name


In [0]:
def run_silver_header_arithmetic_validation(silver_header_df, silver_lines_df, include_warning_rules=True, run_label="pre_route"):
    datasource = get_or_create_spark_datasource(silver_ge_context, f"silver_header_arithmetic_datasource_{run_label}")
    silver_header_arith_df = build_silver_header_arithmetic_df(silver_header_df, silver_lines_df)
    batch_request = build_batch_request(
        datasource=datasource,
        asset_name=f"silver_header_arithmetic_asset_{run_label}",
        dataframe=silver_header_arith_df.select(*SILVER_HEADER_ARITH_COLUMNS),
    )
    suite_name = f"silver_header_arithmetic_suite_{run_label}"
    checkpoint_name = f"silver_header_arithmetic_checkpoint_{run_label}"
    silver_ge_context.add_or_update_expectation_suite(expectation_suite_name=suite_name)
    validator = silver_ge_context.get_validator(batch_request=batch_request, expectation_suite_name=suite_name)

    header_keys = ["InvoiceId", "_source_file", "source_type"]

    for col_name in SILVER_HEADER_ARITH_COLUMNS:
        validator.expect_column_to_exist(col_name)

    validator.expect_column_values_to_be_between(
        "_header_total_diff",
        min_value=-SILVER_WARNING_UPPER_BOUND_HEADER,
        max_value=SILVER_WARNING_UPPER_BOUND_HEADER,
        result_format=build_result_format(header_keys),
        meta=build_issue_meta("silver_header_arithmetic_total_error", "ERROR", "header", "header_total_diff", "InvoiceTotal exceeds the allowed reconciliation error band", header_keys),
    )
    validator.expect_column_values_to_be_between(
        "_line_to_subtotal_diff",
        min_value=-SILVER_WARNING_UPPER_BOUND_HEADER,
        max_value=SILVER_WARNING_UPPER_BOUND_HEADER,
        result_format=build_result_format(header_keys),
        meta=build_issue_meta("silver_header_arithmetic_subtotal_rollup_error", "ERROR", "header", "line_to_subtotal_diff", "SubTotal exceeds the allowed reconciliation error band", header_keys),
    )

    if include_warning_rules:
        validator.expect_column_values_to_be_between(
            "_header_total_diff",
            min_value=-SILVER_ARITH_TOLERANCE_HEADER,
            max_value=SILVER_ARITH_TOLERANCE_HEADER,
            result_format=build_result_format(header_keys),
            meta=build_issue_meta("silver_header_arithmetic_total_warning", "WARNING", "header", "header_total_diff", "InvoiceTotal exceeds the strict reconciliation tolerance", header_keys),
        )
        validator.expect_column_values_to_be_between(
            "_line_to_subtotal_diff",
            min_value=-SILVER_ARITH_TOLERANCE_HEADER,
            max_value=SILVER_ARITH_TOLERANCE_HEADER,
            result_format=build_result_format(header_keys),
            meta=build_issue_meta("silver_header_arithmetic_subtotal_rollup_warning", "WARNING", "header", "line_to_subtotal_diff", "SubTotal exceeds the strict reconciliation tolerance", header_keys),
        )

    validator.save_expectation_suite(discard_failed_expectations=False)
    add_or_update_checkpoint(silver_ge_context, checkpoint_name, batch_request, suite_name)
    started_at = datetime.now(timezone.utc)
    checkpoint_result = silver_ge_context.run_checkpoint(checkpoint_name=checkpoint_name)
    ended_at = datetime.now(timezone.utc)
    validation_result = extract_first_validation(checkpoint_result)
    write_ge_runtime_metric(
        layer="silver",
        suite_name=suite_name,
        run_id=checkpoint_result.run_id,
        started_at=started_at,
        ended_at=ended_at,
        rows_evaluated=silver_header_arith_df.count(),
        validation_result=validation_result
    )
    return checkpoint_result, suite_name


def run_silver_lines_arithmetic_validation(silver_lines_df, include_warning_rules=True, run_label="pre_route"):
    datasource = get_or_create_spark_datasource(silver_ge_context, f"silver_lines_arithmetic_datasource_{run_label}")
    silver_lines_arith_df = build_silver_lines_arithmetic_df(silver_lines_df)
    batch_request = build_batch_request(
        datasource=datasource,
        asset_name=f"silver_lines_arithmetic_asset_{run_label}",
        dataframe=silver_lines_arith_df.select(*SILVER_LINES_ARITH_COLUMNS),
    )
    suite_name = f"silver_lines_arithmetic_suite_{run_label}"
    checkpoint_name = f"silver_lines_arithmetic_checkpoint_{run_label}"
    silver_ge_context.add_or_update_expectation_suite(expectation_suite_name=suite_name)
    validator = silver_ge_context.get_validator(batch_request=batch_request, expectation_suite_name=suite_name)

    line_keys = ["InvoiceId", "LineNumber", "_source_file", "source_type"]

    for col_name in SILVER_LINES_ARITH_COLUMNS:
        validator.expect_column_to_exist(col_name)

    validator.expect_column_values_to_be_between(
        "_line_subtotal_diff",
        min_value=-SILVER_WARNING_UPPER_BOUND_LINE,
        max_value=SILVER_WARNING_UPPER_BOUND_LINE,
        result_format=build_result_format(line_keys),
        meta=build_issue_meta("silver_lines_arithmetic_subtotal_error", "ERROR", "lines", "line_subtotal_diff", "Line subtotal exceeds the allowed reconciliation error band", line_keys),
    )

    if include_warning_rules:
        validator.expect_column_values_to_be_between(
            "_line_subtotal_diff",
            min_value=-SILVER_ARITH_TOLERANCE_LINE,
            max_value=SILVER_ARITH_TOLERANCE_LINE,
            result_format=build_result_format(line_keys),
            meta=build_issue_meta("silver_lines_arithmetic_subtotal_warning", "WARNING", "lines", "line_subtotal_diff", "Line subtotal exceeds the strict reconciliation tolerance", line_keys),
        )

    validator.save_expectation_suite(discard_failed_expectations=False)
    add_or_update_checkpoint(silver_ge_context, checkpoint_name, batch_request, suite_name)
    started_at = datetime.now(timezone.utc)
    checkpoint_result = silver_ge_context.run_checkpoint(checkpoint_name=checkpoint_name)
    ended_at = datetime.now(timezone.utc)
    validation_result = extract_first_validation(checkpoint_result)
    write_ge_runtime_metric(
        layer="silver",
        suite_name=suite_name,
        run_id=checkpoint_result.run_id,
        started_at=started_at,
        ended_at=ended_at,
        rows_evaluated=silver_lines_arith_df.count(),
        validation_result=validation_result
    )
    return checkpoint_result, suite_name

In [0]:
silver_header_df = spark.read.format("delta").load(VALID_SILVER_HEADER_PATH).cache()
silver_lines_df = spark.read.format("delta").load(VALID_SILVER_LINES_PATH).cache()

if silver_header_df.rdd.isEmpty() or silver_lines_df.rdd.isEmpty():
    raise RuntimeError("Silver STOP failure: standardized Silver inputs are empty.")

missing_header = [c for c in SILVER_HEADER_COLUMNS if c not in silver_header_df.columns]
missing_lines = [c for c in SILVER_LINES_COLUMNS if c not in silver_lines_df.columns]
if missing_header or missing_lines:
    raise RuntimeError(
        f"Silver STOP failure: canonical schema is incomplete. "
        f"missing_header={missing_header}, missing_lines={missing_lines}"
    )

# Exclude nulls from duplicate check — null keys are missing values, not true duplicates.
# GE's expect_column_values_to_not_be_null will flag them, and the quarantine join will handle them.
dup_header_df = (silver_header_df
    .filter(col("InvoiceId").isNotNull())
    .groupBy("InvoiceId").count()
    .filter(col("count") > 1))
dup_lines_df = (silver_lines_df
    .filter(col("InvoiceId").isNotNull() & col("LineNumber").isNotNull())
    .groupBy("InvoiceId", "LineNumber").count()
    .filter(col("count") > 1))
duplicate_header_count = dup_header_df.count()
duplicate_line_count = dup_lines_df.count()

if duplicate_header_count > 0:
    display(dup_header_df.join(silver_header_df, on="InvoiceId", how="inner"))
if duplicate_line_count > 0:
    display(dup_lines_df.join(silver_lines_df, on=["InvoiceId", "LineNumber"], how="inner"))
if duplicate_header_count > 0 or duplicate_line_count > 0:
    raise RuntimeError(
        "Silver STOP failure: canonical keys are unstable after transform. "
        f"duplicate_header_count={duplicate_header_count}, "
        f"duplicate_line_count={duplicate_line_count}"
    )

In [0]:
pre_route_silver_header_checkpoint_result, PRE_ROUTE_SILVER_HEADER_SUITE_NAME = run_silver_header_validation(
    silver_header_df, run_label="pre_route"
)
pre_route_silver_lines_checkpoint_result, PRE_ROUTE_SILVER_LINES_SUITE_NAME = run_silver_lines_validation(
    silver_lines_df, run_label="pre_route"
)
pre_route_silver_header_arithmetic_checkpoint_result, PRE_ROUTE_SILVER_HEADER_ARITH_SUITE_NAME = run_silver_header_arithmetic_validation(
    silver_header_df, silver_lines_df, include_warning_rules=True, run_label="pre_route"
)
pre_route_silver_lines_arithmetic_checkpoint_result, PRE_ROUTE_SILVER_LINES_ARITH_SUITE_NAME = run_silver_lines_arithmetic_validation(
    silver_lines_df, include_warning_rules=True, run_label="pre_route"
)

pre_route_validation_results = [
    extract_first_validation(pre_route_silver_header_checkpoint_result),
    extract_first_validation(pre_route_silver_lines_checkpoint_result),
    extract_first_validation(pre_route_silver_header_arithmetic_checkpoint_result),
    extract_first_validation(pre_route_silver_lines_arithmetic_checkpoint_result),
]

pre_route_summaries = [
    summarize_result(pre_route_validation_results[0], PRE_ROUTE_SILVER_HEADER_SUITE_NAME),
    summarize_result(pre_route_validation_results[1], PRE_ROUTE_SILVER_LINES_SUITE_NAME),
    summarize_result(pre_route_validation_results[2], PRE_ROUTE_SILVER_HEADER_ARITH_SUITE_NAME),
    summarize_result(pre_route_validation_results[3], PRE_ROUTE_SILVER_LINES_ARITH_SUITE_NAME),
]

print("Pre-routing GE summaries on full canonical Silver datasets")
for summary in pre_route_summaries:
    display_summary(summary)

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Pre-routing GE summaries on full canonical Silver datasets

  Suite : silver_header_suite_pre_route
  Status: FAILED
  Evaluated : 56  |  Passed: 47  |  Failed: 9  |  Score: 83.9%
  -- Failed Expectations --
   [InvoiceId] expect_column_values_to_not_be_null
      Result: {"element_count": 1251, "unexpected_count": 15, "unexpected_percent": 1.1990407673860912, "partial_unexpected_list": [null, null, null, null, null, null, null, null, null, null, null, null, null, null
   [OrderDate] expect_column_values_to_be_between
      Result: {"element_count": 1251, "unexpected_count": 1, "unexpected_percent": 0.07993605115907274, "partial_unexpected_list": ["2026-05-09"], "missing_count": 0, "missing_percent": 0.0, "unexpected_percent_tot
   [CustomerName] expect_column_values_to_not_be_null
      Result: {"element_count": 1251, "unexpected_count": 16, "unexpected_percent": 1.2789768185451638, "partial_unexpected_list": [null, null, null, null, null, null, null, null, null, null, null, null, nul

In [0]:
ge_issue_events_df = issue_df_from_validation_results(pre_route_validation_results)

deterministic_issue_events_df = empty_issue_df()

deterministic_issue_events_df = deterministic_issue_events_df.unionByName(
    issue_events_from_condition(
        silver_header_df,
        col("OrderDate").isNotNull() & (
            (col("OrderDate") < lit(SILVER_ORDER_DATE_MIN)) |
            (col("OrderDate") > lit(date.today()))
        ),
        dataset="header",
        rule_id="silver_header_order_date_valid",
        severity="ERROR",
        issue_type="contract",
        dq_reason="OrderDate is outside the allowed range",
    ),
    allowMissingColumns=True,
)

deterministic_issue_events_df = deterministic_issue_events_df.unionByName(
    issue_events_from_condition(
        silver_header_df,
        col("CustomerName").isNull(),
        dataset="header",
        rule_id="silver_header_customer_name_present",
        severity="ERROR",
        issue_type="contract",
        dq_reason="CustomerName is missing in Silver header",
    ),
    allowMissingColumns=True,
)

deterministic_issue_events_df = deterministic_issue_events_df.unionByName(
    issue_events_from_condition(
        silver_lines_df,
        col("ProductName").isNull(),
        dataset="lines",
        rule_id="silver_lines_product_name_present",
        severity="ERROR",
        issue_type="contract",
        dq_reason="ProductName is missing in Silver lines",
        line_col="LineNumber",
    ),
    allowMissingColumns=True,
)

deterministic_issue_events_df = deterministic_issue_events_df.unionByName(
    issue_events_from_condition(
        silver_lines_df,
        col("ProductId").isNull(),
        dataset="lines",
        rule_id="silver_lines_product_id_present",
        severity="ERROR",
        issue_type="contract",
        dq_reason="ProductId is missing in Silver lines",
        line_col="LineNumber",
    ),
    allowMissingColumns=True,
)

referential_issue_events_df = empty_issue_df()
orphan_lines_df = silver_lines_df.join(silver_header_df.select("InvoiceId").distinct(), on="InvoiceId", how="left_anti")
header_without_lines_df = silver_header_df.join(silver_lines_df.select("InvoiceId").distinct(), on="InvoiceId", how="left_anti")

referential_issue_events_df = referential_issue_events_df.unionByName(
    orphan_lines_df.select(
        F.lit("silver").alias("layer"),
        F.lit("lines").alias("dataset"),
        F.lit("silver_lines_have_header").alias("rule_id"),
        F.lit("ERROR").alias("severity"),
        F.col("_source_file").cast("string").alias("_source_file"),
        F.col("source_type").cast("string").alias("source_type"),
        F.col("InvoiceId").cast("string").alias("InvoiceId"),
        F.col("LineNumber").cast("string").alias("LineNumber"),
        F.lit("referential_integrity").alias("issue_type"),
        F.lit("Line item does not have a matching header").alias("dq_reason"),
        F.current_timestamp().alias("issue_ts"),
    ).dropDuplicates(),
    allowMissingColumns=True,
)

referential_issue_events_df = referential_issue_events_df.unionByName(
    header_without_lines_df.select(
        F.lit("silver").alias("layer"),
        F.lit("header").alias("dataset"),
        F.lit("silver_header_has_lines").alias("rule_id"),
        F.lit("ERROR").alias("severity"),
        F.col("_source_file").cast("string").alias("_source_file"),
        F.col("source_type").cast("string").alias("source_type"),
        F.col("InvoiceId").cast("string").alias("InvoiceId"),
        F.lit(None).cast("string").alias("LineNumber"),
        F.lit("entity_completeness").alias("issue_type"),
        F.lit("Header does not have any matching lines").alias("dq_reason"),
        F.current_timestamp().alias("issue_ts"),
    ).dropDuplicates(),
    allowMissingColumns=True,
)

contract_issue_events_df = (
    ge_issue_events_df
    .unionByName(deterministic_issue_events_df, allowMissingColumns=True)
    .unionByName(referential_issue_events_df, allowMissingColumns=True)
    .dropDuplicates()
)

contract_error_invoice_ids_df = contract_issue_events_df.filter(col("severity") == "ERROR").select("InvoiceId").distinct()
contract_warning_invoice_ids_df = (
    contract_issue_events_df.filter(col("severity") == "WARNING").select("InvoiceId").distinct()
    .join(contract_error_invoice_ids_df, on="InvoiceId", how="left_anti")
)


In [0]:
def classify_line_diff_severity(diff_col):
    abs_diff = F.abs(diff_col)
    return (
        when(diff_col.isNull(), F.lit("PASS"))
        .when(abs_diff <= F.lit(SILVER_ARITH_TOLERANCE_LINE), F.lit("PASS"))
        .when(abs_diff <= F.lit(SILVER_WARNING_UPPER_BOUND_LINE), F.lit("WARNING"))
        .otherwise(F.lit("ERROR"))
    )


def classify_header_diff_severity(diff_col):
    abs_diff = F.abs(diff_col)
    return (
        when(diff_col.isNull(), F.lit("PASS"))
        .when(abs_diff <= F.lit(SILVER_ARITH_TOLERANCE_HEADER), F.lit("PASS"))
        .when(abs_diff <= F.lit(SILVER_WARNING_UPPER_BOUND_HEADER), F.lit("WARNING"))
        .otherwise(F.lit("ERROR"))
    )


In [0]:
silver_lines_arith_df = build_silver_lines_arithmetic_df(silver_lines_df)
silver_header_arith_df = build_silver_header_arithmetic_df(silver_header_df, silver_lines_df)

arithmetic_issue_events_df = (
    silver_lines_arith_df
    .withColumn("severity", classify_line_diff_severity(col("_line_subtotal_diff")))
    .filter(col("severity") != "PASS")
    .select(
        F.lit("silver").alias("layer"),
        F.lit("lines").alias("dataset"),
        F.lit("silver_lines_arithmetic_subtotal").alias("rule_id"),
        col("severity"),
        col("_source_file").cast("string").alias("_source_file"),
        col("source_type").cast("string").alias("source_type"),
        col("InvoiceId").cast("string").alias("InvoiceId"),
        col("LineNumber").cast("string").alias("LineNumber"),
        F.lit("line_subtotal_diff").alias("issue_type"),
        F.lit("Line subtotal does not match Quantity * UnitPrice").alias("dq_reason"),
        F.current_timestamp().alias("issue_ts"),
    )
    .unionByName(
        silver_header_arith_df
        .withColumn("severity", classify_header_diff_severity(col("_header_total_diff")))
        .filter(col("severity") != "PASS")
        .select(
            F.lit("silver").alias("layer"),
            F.lit("header").alias("dataset"),
            F.lit("silver_header_arithmetic_total").alias("rule_id"),
            col("severity"),
            col("_source_file").cast("string").alias("_source_file"),
            col("source_type").cast("string").alias("source_type"),
            col("InvoiceId").cast("string").alias("InvoiceId"),
            F.lit(None).cast("string").alias("LineNumber"),
            F.lit("header_total_diff").alias("issue_type"),
            F.lit("InvoiceTotal does not reconcile with header arithmetic").alias("dq_reason"),
            F.current_timestamp().alias("issue_ts"),
        ),
        allowMissingColumns=True,
    )
    .unionByName(
        silver_header_arith_df
        .withColumn("severity", classify_header_diff_severity(col("_line_to_subtotal_diff")))
        .filter(col("severity") != "PASS")
        .select(
            F.lit("silver").alias("layer"),
            F.lit("header").alias("dataset"),
            F.lit("silver_header_arithmetic_subtotal_rollup").alias("rule_id"),
            col("severity"),
            col("_source_file").cast("string").alias("_source_file"),
            col("source_type").cast("string").alias("source_type"),
            col("InvoiceId").cast("string").alias("InvoiceId"),
            F.lit(None).cast("string").alias("LineNumber"),
            F.lit("line_to_subtotal_diff").alias("issue_type"),
            F.lit("SubTotal does not reconcile to sum(ItemSubTotal)").alias("dq_reason"),
            F.current_timestamp().alias("issue_ts"),
        ),
        allowMissingColumns=True,
    )
    .dropDuplicates()
)

arithmetic_error_invoice_ids_df = arithmetic_issue_events_df.filter(col("severity") == "ERROR").select("InvoiceId").distinct()
arithmetic_warning_invoice_ids_df = (
    arithmetic_issue_events_df.filter(col("severity") == "WARNING").select("InvoiceId").distinct()
    .join(arithmetic_error_invoice_ids_df, on="InvoiceId", how="left_anti")
)

final_error_invoice_ids_df = contract_error_invoice_ids_df.unionByName(arithmetic_error_invoice_ids_df).distinct()
final_warning_invoice_ids_df = (
    contract_warning_invoice_ids_df.unionByName(arithmetic_warning_invoice_ids_df).distinct()
    .join(final_error_invoice_ids_df, on="InvoiceId", how="left_anti")
)

warning_rule_rollup_df = (
    contract_issue_events_df.filter(col("severity") == "WARNING").select("InvoiceId", "rule_id", "dq_reason")
    .unionByName(arithmetic_issue_events_df.filter(col("severity") == "WARNING").select("InvoiceId", "rule_id", "dq_reason"), allowMissingColumns=True)
    .groupBy("InvoiceId")
    .agg(
        F.collect_set("rule_id").alias("dq_warning_rules"),
        F.collect_set("dq_reason").alias("dq_warning_reasons"),
    )
)

In [0]:
non_null_final_error_invoice_ids_df = final_error_invoice_ids_df.filter(col("InvoiceId").isNotNull()).distinct()
non_null_final_warning_invoice_ids_df = final_warning_invoice_ids_df.filter(col("InvoiceId").isNotNull()).distinct()
warning_rule_rollup_non_null_df = warning_rule_rollup_df.filter(col("InvoiceId").isNotNull())

null_key_header_df = silver_header_df.filter(col("InvoiceId").isNull())
null_key_lines_df = silver_lines_df.filter(col("InvoiceId").isNull())

valid_silver_header_df = (
    silver_header_df
    .filter(col("InvoiceId").isNotNull())
    .join(non_null_final_error_invoice_ids_df, on="InvoiceId", how="left_anti")
    .join(warning_rule_rollup_non_null_df, on="InvoiceId", how="left")
    .withColumn("dq_has_warning", F.when(col("dq_warning_rules").isNotNull(), F.lit(True)).otherwise(F.lit(False)))
)
valid_silver_lines_df = (
    silver_lines_df
    .filter(col("InvoiceId").isNotNull())
    .join(non_null_final_error_invoice_ids_df, on="InvoiceId", how="left_anti")
    .join(warning_rule_rollup_non_null_df, on="InvoiceId", how="left")
    .withColumn("dq_has_warning", F.when(col("dq_warning_rules").isNotNull(), F.lit(True)).otherwise(F.lit(False)))
)

quarantine_silver_header_df = (
    silver_header_df.join(non_null_final_error_invoice_ids_df, on="InvoiceId", how="inner")
    .unionByName(null_key_header_df, allowMissingColumns=True)
    .dropDuplicates()
)
quarantine_silver_lines_df = (
    silver_lines_df.join(non_null_final_error_invoice_ids_df, on="InvoiceId", how="inner")
    .unionByName(null_key_lines_df, allowMissingColumns=True)
    .dropDuplicates()
)
warning_silver_header_df = silver_header_df.join(non_null_final_warning_invoice_ids_df, on="InvoiceId", how="inner").join(warning_rule_rollup_non_null_df, on="InvoiceId", how="left")
warning_silver_lines_df = silver_lines_df.join(non_null_final_warning_invoice_ids_df, on="InvoiceId", how="inner").join(warning_rule_rollup_non_null_df, on="InvoiceId", how="left")

silver_issue_events_df = contract_issue_events_df.unionByName(arithmetic_issue_events_df, allowMissingColumns=True).dropDuplicates()


In [0]:
final_silver_header_checkpoint_result, FINAL_SILVER_HEADER_SUITE_NAME = run_silver_header_validation(
    valid_silver_header_df, include_warning_rules=False, run_label="final_valid"
)
final_silver_lines_checkpoint_result, FINAL_SILVER_LINES_SUITE_NAME = run_silver_lines_validation(
    valid_silver_lines_df, include_warning_rules=False, run_label="final_valid"
)
final_silver_header_arithmetic_checkpoint_result, FINAL_SILVER_HEADER_ARITH_SUITE_NAME = run_silver_header_arithmetic_validation(
    valid_silver_header_df, valid_silver_lines_df, include_warning_rules=False, run_label="final_valid"
)
final_silver_lines_arithmetic_checkpoint_result, FINAL_SILVER_LINES_ARITH_SUITE_NAME = run_silver_lines_arithmetic_validation(
    valid_silver_lines_df, include_warning_rules=False, run_label="final_valid"
)

final_summaries = [
    summarize_result(extract_first_validation(final_silver_header_checkpoint_result), FINAL_SILVER_HEADER_SUITE_NAME),
    summarize_result(extract_first_validation(final_silver_lines_checkpoint_result), FINAL_SILVER_LINES_SUITE_NAME),
    summarize_result(extract_first_validation(final_silver_header_arithmetic_checkpoint_result), FINAL_SILVER_HEADER_ARITH_SUITE_NAME),
    summarize_result(extract_first_validation(final_silver_lines_arithmetic_checkpoint_result), FINAL_SILVER_LINES_ARITH_SUITE_NAME),
]

print("Post-routing GE summaries on valid Silver outputs")
for summary in final_summaries:
    display_summary(summary)


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Post-routing GE summaries on valid Silver outputs

  Suite : silver_header_suite_final_valid
  Status: PASSED
  Evaluated : 52  |  Passed: 52  |  Failed: 0  |  Score: 100.0%


  Suite : silver_lines_suite_final_valid
  Status: PASSED
  Evaluated : 37  |  Passed: 37  |  Failed: 0  |  Score: 100.0%


  Suite : silver_header_arithmetic_suite_final_valid
  Status: PASSED
  Evaluated : 16  |  Passed: 16  |  Failed: 0  |  Score: 100.0%


  Suite : silver_lines_arithmetic_suite_final_valid
  Status: PASSED
  Evaluated : 10  |  Passed: 10  |  Failed: 0  |  Score: 100.0%



In [0]:
total_invoice_count = silver_header_df.select("InvoiceId").distinct().count()
error_invoice_count = final_error_invoice_ids_df.count()
warning_invoice_count = final_warning_invoice_ids_df.count()
error_rate = (error_invoice_count / total_invoice_count) if total_invoice_count else 0.0

print(f"Silver invoices evaluated    = {total_invoice_count:,}")
print(f"Silver invoices quarantined  = {error_invoice_count:,}")
print(f"Silver invoices with warning = {warning_invoice_count:,}")
print(f"Silver error rate            = {error_rate:.4%}")

pre_route_failed_suites = [s["suite"] for s in pre_route_summaries if not s["success"]]
if pre_route_failed_suites:
    print("Pre-routing GE found issues in canonical Silver data:")
    print(", ".join(pre_route_failed_suites))
else:
    print("Pre-routing GE found no canonical Silver issues.")

if any(not summary["success"] for summary in final_summaries):
    failed_suites = [s["suite"] for s in final_summaries if not s["success"]]
    raise RuntimeError(
        "Silver validation FAILED after routing - downstream Gold / serving processing halted. "
        f"Failed suites: {', '.join(failed_suites)}"
    )

if SILVER_ENABLE_ARITH_PIPELINE_GATE and (
    error_invoice_count > SILVER_GATE_MAX_ERROR_INVOICES or error_rate > SILVER_GATE_MAX_ERROR_RATE
):
    raise RuntimeError(
        "Silver ERROR-volume gate FAILED - downstream Gold / serving processing halted. "
        f"error_invoice_count={error_invoice_count}, error_rate={error_rate:.4%}, "
        f"thresholds=(max_invoices={SILVER_GATE_MAX_ERROR_INVOICES}, max_rate={SILVER_GATE_MAX_ERROR_RATE:.2%})"
    )

Silver invoices evaluated    = 1,237
Silver invoices quarantined  = 152
Silver invoices with warning = 227
Silver error rate            = 12.2878%
Pre-routing GE found issues in canonical Silver data:
silver_header_suite_pre_route, silver_lines_suite_pre_route, silver_header_arithmetic_suite_pre_route, silver_lines_arithmetic_suite_pre_route


In [0]:
valid_silver_header_df.write.format("delta").mode("overwrite").save(GE_VALID_SILVER_HEADER_PATH)
valid_silver_lines_df.write.format("delta").mode("overwrite").save(GE_VALID_SILVER_LINES_PATH)
quarantine_silver_header_df.write.format("delta").mode("overwrite").save(GE_QUARANTINE_SILVER_HEADER_PATH)
quarantine_silver_lines_df.write.format("delta").mode("overwrite").save(GE_QUARANTINE_SILVER_LINES_PATH)
warning_silver_header_df.write.format("delta").mode("overwrite").save(GE_WARNING_SILVER_HEADER_PATH)
warning_silver_lines_df.write.format("delta").mode("overwrite").save(GE_WARNING_SILVER_LINES_PATH)
silver_issue_events_df.write.format("delta").mode("append").save(GE_ISSUE_LOG_PATH)

# Build and publish Silver Data Docs after all checkpoints have run
ensure_data_docs_site(silver_ge_context)

silver_data_docs = publish_data_docs(silver_ge_context)
render_data_docs_preview(silver_data_docs, "Silver Great Expectations Data Docs")


Data Docs root: dbfs:/FileStore/great_expectations/silver/
 - dbfs:/FileStore/great_expectations/silver/expectations/
 - dbfs:/FileStore/great_expectations/silver/index.html
 - dbfs:/FileStore/great_expectations/silver/static/
 - dbfs:/FileStore/great_expectations/silver/validations/

Data Docs expectations: dbfs:/FileStore/great_expectations/silver/expectations/
 - dbfs:/FileStore/great_expectations/silver/expectations/silver_header_arithmetic_suite_final_valid.html
 - dbfs:/FileStore/great_expectations/silver/expectations/silver_header_arithmetic_suite_pre_route.html
 - dbfs:/FileStore/great_expectations/silver/expectations/silver_header_suite_final_valid.html
 - dbfs:/FileStore/great_expectations/silver/expectations/silver_header_suite_pre_route.html
 - dbfs:/FileStore/great_expectations/silver/expectations/silver_lines_arithmetic_suite_final_valid.html
 - dbfs:/FileStore/great_expectations/silver/expectations/silver_lines_arithmetic_suite_pre_route.html
 - dbfs:/FileStore/great_ex

Silver Great Expectations Data Docs 
 Open Data Docs index

In [0]:
# Databricks table registration for Metaplane: Silver GE outputs
CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "invoice"

BATCH_SILVER_VALIDATED_HEADER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_silver_validated_header"
BATCH_SILVER_VALIDATED_LINES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_silver_validated_lines"
BATCH_SILVER_QUARANTINE_HEADER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_silver_quarantine_header"
BATCH_SILVER_QUARANTINE_LINES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_silver_quarantine_lines"
BATCH_SILVER_WARNING_HEADER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_silver_warning_header"
BATCH_SILVER_WARNING_LINES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_silver_warning_lines"
BATCH_SILVER_GE_ISSUE_LOG_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_silver_ge_issue_log"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_SILVER_VALIDATED_HEADER_TABLE}
USING DELTA
LOCATION "{GE_VALID_SILVER_HEADER_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_SILVER_VALIDATED_LINES_TABLE}
USING DELTA
LOCATION "{GE_VALID_SILVER_LINES_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_SILVER_QUARANTINE_HEADER_TABLE}
USING DELTA
LOCATION "{GE_QUARANTINE_SILVER_HEADER_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_SILVER_QUARANTINE_LINES_TABLE}
USING DELTA
LOCATION "{GE_QUARANTINE_SILVER_LINES_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_SILVER_WARNING_HEADER_TABLE}
USING DELTA
LOCATION "{GE_WARNING_SILVER_HEADER_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_SILVER_WARNING_LINES_TABLE}
USING DELTA
LOCATION "{GE_WARNING_SILVER_LINES_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_SILVER_GE_ISSUE_LOG_TABLE}
USING DELTA
LOCATION "{GE_ISSUE_LOG_PATH}"
''')

display(spark.sql(f"DESCRIBE DETAIL {BATCH_SILVER_GE_ISSUE_LOG_TABLE}"))


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,0a4f8f11-ce0f-41db-8d97-93c664c53d30,hant-catalog.invoice.batch_silver_ge_issue_log,null,abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/silver/ge/issue_log,2026-04-23T19:50:33.686Z,2026-04-23T19:50:38Z,List(),List(),1,14356,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
dbutils.jobs.taskValues.set(key="ge_valid_silver_header_path", value=GE_VALID_SILVER_HEADER_PATH)
dbutils.jobs.taskValues.set(key="ge_valid_silver_lines_path", value=GE_VALID_SILVER_LINES_PATH)
dbutils.jobs.taskValues.set(key="ge_quarantine_silver_header_path", value=GE_QUARANTINE_SILVER_HEADER_PATH)
dbutils.jobs.taskValues.set(key="ge_quarantine_silver_lines_path", value=GE_QUARANTINE_SILVER_LINES_PATH)
dbutils.jobs.taskValues.set(key="ge_warning_silver_header_path", value=GE_WARNING_SILVER_HEADER_PATH)
dbutils.jobs.taskValues.set(key="ge_warning_silver_lines_path", value=GE_WARNING_SILVER_LINES_PATH)
dbutils.jobs.taskValues.set(key="silver_issue_log_path", value=GE_ISSUE_LOG_PATH)